In [ ]:

# Smart Customer Feedback Analyzer
# Step 1: Install required libraries

!pip -q install datasets transformers sentence-transformers scikit-learn pandas matplotlib seaborn gradio

print("✅ Required libraries installed successfully!")


In [ ]:

# Step 2: Explore the dataset

from datasets import load_dataset
import pandas as pd

DATASET_NAME = "clapAI/MultiLingualSentiment"

# Stream the data without downloading the entire dataset
stream = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True
)

# Load a small sample for inspection
sample = list(stream.shuffle(seed=42, buffer_size=1000).take(2000))

df_sample = pd.DataFrame(sample)

print("Dataset sample loaded successfully!")
print("Number of rows:", len(df_sample))
print("Columns:", df_sample.columns.tolist())

print("\nLanguages:")
print(df_sample["language"].value_counts().head(10))

print("\nSentiment labels:")
print(df_sample["label"].value_counts())

print("\nAvailable domains:")
print(df_sample["domain"].value_counts().head(10))

display(df_sample.head())


In [ ]:

# Step 3: Inspect dataset splits

from datasets import load_dataset
import pandas as pd

for split_name in ["train", "validation", "test"]:

    dataset_stream = load_dataset(
        "clapAI/MultiLingualSentiment",
        split=split_name,
        streaming=True
    )

    sample = list(
        dataset_stream.shuffle(
            seed=42,
            buffer_size=1000
        ).take(500)
    )

    df_check = pd.DataFrame(sample)

    print(f"\n===== {split_name.upper()} =====")

    print("Sentiment labels:")
    print(df_check["label"].value_counts())

    print("\nArabic and English records:")
    print(
        df_check["language"]
        .value_counts()
        .reindex(["ar", "en"], fill_value=0)
    )


In [ ]:

# Step 4: Inspect labels at different dataset positions

import requests

DATASET_NAME = "clapAI/MultiLingualSentiment"

split_sizes = {
    "train": 3147478,
    "validation": 393435,
    "test": 393436
}

API_URL = "https://datasets-server.huggingface.co/rows"

for split_name, total_rows in split_sizes.items():

    print(f"\n===== {split_name.upper()} =====")

    for position in [0.05, 0.25, 0.50, 0.75, 0.95]:

        offset = int(total_rows * position)

        response = requests.get(
            API_URL,
            params={
                "dataset": DATASET_NAME,
                "config": "default",
                "split": split_name,
                "offset": offset,
                "length": 10
            },
            timeout=45
        )

        response.raise_for_status()
        records = [
            item["row"]
            for item in response.json()["rows"]
        ]

        df_part = pd.DataFrame(records)

        print(
            f"{int(position * 100)}%:",
            df_part["label"].value_counts().to_dict()
        )

print("\n✅ Inspection completed!")


In [ ]:

# Step 5: Alternative dataset sampling method

import requests
import pandas as pd

API_URL = "https://datasets-server.huggingface.co/rows"
DATASET_NAME = "clapAI/MultiLingualSentiment"

TRAIN_SIZE = 3147478

positions = {
    "positive": 0.25,
    "neutral": 0.50,
    "negative": 0.75
}

samples = []

for label, position in positions.items():

    offset = int(TRAIN_SIZE * position)

    response = requests.get(
        API_URL,
        params={
            "dataset": DATASET_NAME,
            "config": "default",
            "split": "train",
            "offset": offset,
            "length": 100
        },
        timeout=30
    )

    response.raise_for_status()

    rows = [
        item["row"]
        for item in response.json()["rows"]
    ]

    samples.extend(rows)

df_test = pd.DataFrame(samples)

# Keep only Arabic and English
df_test = df_test[
    df_test["language"].isin(["ar", "en"])
].copy()

print("Total Arabic & English records:", len(df_test))

print("\nDistribution by language and sentiment:")
print(pd.crosstab(
    df_test["language"],
    df_test["label"]
))

print("\nSample reviews:")
display(df_test[["text", "label", "language"]].head())

print("\n✅ Sampling test completed!")


In [ ]:

# Step 6: Build a balanced bilingual dataset

import requests
import pandas as pd
import numpy as np
import time

DATASET = "clapAI/MultiLingualSentiment"
API = "https://datasets-server.huggingface.co/rows"

SPLIT_SIZES = {
    "train": 3147478,
    "validation": 393435,
    "test": 393436
}

LABEL_REGIONS = {
    "positive": (0.15, 0.28),
    "neutral": (0.46, 0.54),
    "negative": (0.75, 0.88)
}

TARGETS = {
    "train": 25,
    "validation": 8,
    "test": 8
}

rng = np.random.default_rng(42)
datasets_final = {}

for split, size in SPLIT_SIZES.items():

    collected = []
    seen = set()
    target = TARGETS[split]

    for label, (start, end) in LABEL_REGIONS.items():

        counts = {"ar": 0, "en": 0}

        for attempt in range(25):

            if all(counts[x] >= target for x in counts):
                break

            offset = int(
                rng.integers(
                    int(size * start),
                    int(size * end) - 100
                )
            )

            try:
                response = requests.get(
                    API,
                    params={
                        "dataset": DATASET,
                        "config": "default",
                        "split": split,
                        "offset": offset,
                        "length": 100
                    },
                    timeout=25
                )
                response.raise_for_status()
                rows = response.json()["rows"]

            except requests.exceptions.RequestException:
                print("Request failed, trying another position...")
                time.sleep(1)
                continue

            for item in rows:
                row = item["row"]
                lang = row.get("language")
                text = str(row.get("text") or "").strip()

                if (
                    lang not in counts
                    or row.get("label") != label
                    or counts[lang] >= target
                    or len(text) < 15
                    or text in seen
                ):
                    continue

                collected.append({
                    "text": text,
                    "label": label,
                    "language": lang,
                    "domain": row.get("domain"),
                    "source": row.get("source")
                })

                seen.add(text)
                counts[lang] += 1

        print(split, label, counts)

    df = pd.DataFrame(collected)

    # Save each split separately
    df.to_csv(
        f"customer_{split}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    datasets_final[split] = df

    print(f"Saved {split}: {len(df)} rows\n")

print("===== FINAL DATA DISTRIBUTION =====")

for split, df in datasets_final.items():
    print(f"\n{split.upper()}")
    if not df.empty:
        print(pd.crosstab(df["language"], df["label"]))

print("\n✅ Dataset extraction completed!")


In [ ]:

# Step 7: Data Quality & Leakage Check

import pandas as pd

train_df = pd.read_csv("customer_train.csv")
val_df = pd.read_csv("customer_validation.csv")
test_df = pd.read_csv("customer_test.csv")

splits = {
    "Train": train_df,
    "Validation": val_df,
    "Test": test_df
}

# Check missing values and duplicates
for name, df in splits.items():
    print(f"\n===== {name} =====")
    print("Rows:", len(df))
    print("Missing texts:", df["text"].isna().sum())
    print("Duplicate texts:", df["text"].duplicated().sum())

# Check overlap between datasets
def normalized_texts(df):
    return set(
        df["text"].dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )

train_texts = normalized_texts(train_df)
val_texts = normalized_texts(val_df)
test_texts = normalized_texts(test_df)

print("\n===== DATA LEAKAGE CHECK =====")
print("Train vs Validation:", len(train_texts & val_texts))
print("Train vs Test:", len(train_texts & test_texts))
print("Validation vs Test:", len(val_texts & test_texts))

# Check review domains
print("\n===== TRAIN DOMAINS =====")
print(train_df["domain"].value_counts().head(15))

# Verify labels and languages
print("\n===== TRAIN DISTRIBUTION =====")
print(pd.crosstab(train_df["language"], train_df["label"]))

print("\n✅ Data quality check completed!")


In [ ]:

# Step 8: Save prepared datasets to Google Drive

from google.colab import drive
from pathlib import Path
import shutil

# Connect Google Drive
drive.mount("/content/drive")

# Create a project folder
project_dir = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)
data_dir = project_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# Save all three CSV files
for filename in [
    "customer_train.csv",
    "customer_validation.csv",
    "customer_test.csv"
]:
    source = Path(filename)
    destination = data_dir / filename

    shutil.copy2(source, destination)
    print(f"Saved: {destination.name}")

# Verify saved files
print("\n===== SAVED DATASETS =====")
for file in data_dir.glob("*.csv"):
    print(file.name, "-", file.stat().st_size, "bytes")

print("\n✅ All datasets backed up successfully!")


In [ ]:

# Step 9: Train Baseline Model
# TF-IDF + Logistic Regression

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# Load prepared datasets
train_df = pd.read_csv(
    data_dir / "customer_train.csv"
)

val_df = pd.read_csv(
    data_dir / "customer_validation.csv"
)

# Training and validation data
X_train = train_df["text"].fillna("")
y_train = train_df["label"]

X_val = val_df["text"].fillna("")
y_val = val_df["label"]

# Build baseline pipeline
baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 2),
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

# Train the model
baseline_model.fit(X_train, y_train)

# Predict validation labels
val_predictions = baseline_model.predict(X_val)

# Evaluate model
baseline_accuracy = accuracy_score(
    y_val, val_predictions
)

baseline_f1 = f1_score(
    y_val,
    val_predictions,
    average="macro",
    zero_division=0
)

print("===== BASELINE VALIDATION RESULTS =====")

print(f"Accuracy: {baseline_accuracy:.4f}")
print(f"Macro F1: {baseline_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_predictions,
        zero_division=0
    )
)

print("\n✅ Baseline model trained successfully!")


In [ ]:

# Step 10: Train Improved Model
# Multilingual Embeddings + Logistic Regression

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch

# Load multilingual embedding model
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    model_name,
    device=device
)

print("Using device:", device)

# Convert text to semantic embeddings
print("Encoding training texts...")

X_train_embeddings = embedding_model.encode(
    train_df["text"].fillna("").tolist(),
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Encoding validation texts...")

X_val_embeddings = embedding_model.encode(
    val_df["text"].fillna("").tolist(),
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Train classifier
improved_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

improved_model.fit(
    X_train_embeddings,
    train_df["label"]
)

# Predict
improved_predictions = improved_model.predict(
    X_val_embeddings
)

# Evaluation
improved_accuracy = accuracy_score(
    val_df["label"],
    improved_predictions
)

improved_f1 = f1_score(
    val_df["label"],
    improved_predictions,
    average="macro",
    zero_division=0
)

print("\n===== IMPROVED MODEL RESULTS =====")

print(f"Accuracy: {improved_accuracy:.4f}")
print(f"Macro F1: {improved_f1:.4f}")

print("\nClassification Report:")

print(
    classification_report(
        val_df["label"],
        improved_predictions,
        zero_division=0
    )
)

# Compare baseline vs improved
comparison = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "Embeddings + Logistic Regression"
    ],
    "Accuracy": [
        baseline_accuracy,
        improved_accuracy
    ],
    "Macro F1": [
        baseline_f1,
        improved_f1
    ]
})

print("\n===== MODEL COMPARISON =====")
display(comparison)

print("\n✅ Improved model experiment completed!")


In [ ]:

# Step 11: Try character-level TF-IDF
# Compare with our previous models

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

results = []

for c_value in [0.1, 1.0, 10.0]:

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(2, 5),
                max_features=20000,
                sublinear_tf=True
            )
        ),
        (
            "classifier",
            LogisticRegression(
                C=c_value,
                max_iter=1000,
                random_state=42
            )
        )
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    results.append({
        "C": c_value,
        "Accuracy": accuracy_score(y_val, predictions),
        "Macro F1": f1_score(
            y_val,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Model": model
    })

# Find the best configuration using validation Macro F1
best_result = max(results, key=lambda x: x["Macro F1"])
char_model = best_result["Model"]

print("===== CHARACTER MODEL RESULTS =====")

for result in results:
    print(
        f"C={result['C']} | "
        f"Accuracy={result['Accuracy']:.4f} | "
        f"Macro F1={result['Macro F1']:.4f}"
    )

print("\n===== BEST CHARACTER MODEL =====")
print("Best C:", best_result["C"])
print(f"Accuracy: {best_result['Accuracy']:.4f}")
print(f"Macro F1: {best_result['Macro F1']:.4f}")

print("\n===== ALL MODEL COMPARISON =====")

comparison = pd.DataFrame([
    {
        "Model": "Word TF-IDF Baseline",
        "Accuracy": baseline_accuracy,
        "Macro F1": baseline_f1
    },
    {
        "Model": "Multilingual Embeddings",
        "Accuracy": improved_accuracy,
        "Macro F1": improved_f1
    },
    {
        "Model": "Character TF-IDF",
        "Accuracy": best_result["Accuracy"],
        "Macro F1": best_result["Macro F1"]
    }
])

display(comparison)

print("\n✅ Third experiment completed!")


In [ ]:

# Step 12: Semantic Clustering with K-Means

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Use the training embeddings from Step 10
X_cluster = X_train_embeddings.copy()

# Normalize embeddings
X_cluster = X_cluster / np.maximum(
    np.linalg.norm(X_cluster, axis=1, keepdims=True),
    1e-12
)

# Group reviews into 3 semantic clusters
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_cluster)

cluster_df = train_df.copy()
cluster_df["cluster"] = cluster_labels

# Evaluate cluster separation
silhouette = silhouette_score(
    X_cluster,
    cluster_labels,
    metric="cosine"
)

print("===== CLUSTERING RESULTS =====")
print("Number of clusters:", 3)
print("Silhouette Score:", round(silhouette, 4))

print("\n===== CLUSTER DISTRIBUTION =====")
print(cluster_df["cluster"].value_counts().sort_index())

print("\n===== CLUSTER BY LANGUAGE =====")
print(pd.crosstab(
    cluster_df["cluster"],
    cluster_df["language"]
))

print("\n===== CLUSTER BY SENTIMENT =====")
print(pd.crosstab(
    cluster_df["cluster"],
    cluster_df["label"]
))

# Show 3 representative reviews per cluster
print("\n===== REPRESENTATIVE REVIEWS =====")

for cluster_id in range(3):

    indices = np.where(cluster_labels == cluster_id)[0]

    center = kmeans.cluster_centers_[cluster_id]
    center = center / max(np.linalg.norm(center), 1e-12)

    similarities = X_cluster[indices] @ center

    top_indices = indices[np.argsort(similarities)[-3:][::-1]]

    print(f"\n--- Cluster {cluster_id} ---")

    for idx in top_indices:
        row = cluster_df.iloc[idx]
        print("Language:", row["language"])
        print("Sentiment:", row["label"])
        print("Domain:", row["domain"])
        print("Review:", str(row["text"])[:300])
        print("---")

print("\n✅ Semantic clustering completed!")


In [ ]:

# Step 13: Improved Customer Feedback Clustering
# Filter customer reviews + cluster each language separately

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Step A: Keep customer-feedback-related domains
customer_keywords = [
    "hotel",
    "restaurant",
    "amazon",
    "product",
    "company",
    "app reviews",
    "airlines",
    "places reviews"
]

customer_df = train_df[
    train_df["domain"].fillna("").str.lower().apply(
        lambda d: any(word in d for word in customer_keywords)
    )
].copy()

customer_df = customer_df.reset_index(drop=True)

print("Customer feedback records:", len(customer_df))

print("\nCustomer feedback by language:")
print(customer_df["language"].value_counts())

# Step B: Generate embeddings
customer_embeddings = embedding_model.encode(
    customer_df["text"].fillna("").tolist(),
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Step C: Cluster separately for Arabic and English
cluster_results = []

for language in ["ar", "en"]:

    language_indices = np.where(
        customer_df["language"].values == language
    )[0]

    X_lang = customer_embeddings[language_indices]

    if len(language_indices) < 4:
        print(f"Not enough records for {language}")
        continue

    n_clusters = 2

    km = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10
    )

    labels = km.fit_predict(X_lang)

    if len(set(labels)) > 1:
        score = silhouette_score(
            X_lang,
            labels,
            metric="cosine"
        )
    else:
        score = float("nan")

    print(f"\n===== {language.upper()} CLUSTERING =====")
    print("Records:", len(language_indices))
    print("Silhouette Score:", round(score, 4))

    for cluster_id in range(n_clusters):

        local_indices = np.where(labels == cluster_id)[0]

        if len(local_indices) == 0:
            continue

        center = km.cluster_centers_[cluster_id]
        center = center / max(
            np.linalg.norm(center), 1e-12
        )

        similarities = X_lang[local_indices] @ center

        best_local = local_indices[
            np.argsort(similarities)[::-1][:3]
        ]

        print(f"\n--- Cluster {cluster_id} ---")
        print("Number of reviews:", len(local_indices))

        cluster_rows = customer_df.iloc[
            language_indices[local_indices]
        ]

        print(
            "Sentiments:",
            cluster_rows["label"].value_counts().to_dict()
        )

        print("Representative reviews:")

        for local_idx in best_local:
            global_idx = language_indices[local_idx]
            review = customer_df.iloc[global_idx]

            print("-", str(review["text"])[:350])

        for local_idx in local_indices:
            global_idx = language_indices[local_idx]

            cluster_results.append({
                "text": customer_df.iloc[global_idx]["text"],
                "language": language,
                "sentiment": customer_df.iloc[global_idx]["label"],
                "domain": customer_df.iloc[global_idx]["domain"],
                "cluster": f"{language}_{cluster_id}"
            })

improved_cluster_df = pd.DataFrame(cluster_results)

print("\n===== FINAL CLUSTER COUNTS =====")

if not improved_cluster_df.empty:
    print(improved_cluster_df["cluster"].value_counts())

print("\n✅ Improved clustering experiment completed!")


In [ ]:

# Step 14: Generate evidence-based customer insights

import pandas as pd
import numpy as np
from pathlib import Path

insights = []

for cluster_id, group in improved_cluster_df.groupby("cluster"):

    group = group.copy()

    # Obtain embeddings for reviews in this cluster
    vectors = embedding_model.encode(
        group["text"].fillna("").tolist(),
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    # Select the review closest to the cluster mean
    center = vectors.mean(axis=0)
    center = center / max(np.linalg.norm(center), 1e-12)

    similarities = vectors @ center
    representative_index = int(np.argmax(similarities))

    sentiment_counts = group["sentiment"].value_counts()

    insights.append({
        "Cluster": cluster_id,
        "Language": group["language"].iloc[0],
        "Total Reviews": len(group),
        "Positive": int(sentiment_counts.get("positive", 0)),
        "Neutral": int(sentiment_counts.get("neutral", 0)),
        "Negative": int(sentiment_counts.get("negative", 0)),
        "Negative Rate (%)": round(
            100 * sentiment_counts.get("negative", 0) / len(group),
            1
        ),
        "Representative Review": group.iloc[
            representative_index
        ]["text"]
    })

insights_df = pd.DataFrame(insights).sort_values("Cluster")

print("===== CUSTOMER INSIGHTS SUMMARY =====")

display(
    insights_df.drop(columns=["Representative Review"])
)

print("\n===== REPRESENTATIVE FEEDBACK =====")

for _, row in insights_df.iterrows():
    print(f"\nCluster: {row['Cluster']}")
    print(f"Language: {row['Language']}")
    print(f"Negative Rate: {row['Negative Rate (%)']}%")
    print("Representative Review:")
    print(str(row["Representative Review"])[:500])

# Save results permanently to Google Drive
results_dir = project_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)

insights_df.to_csv(
    results_dir / "customer_insights.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ Customer insights generated and saved!")


In [ ]:

# Step 15: Final Test Evaluation

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

# Load held-out test data
test_df = pd.read_csv(
    data_dir / "customer_test.csv"
)

X_test = test_df["text"].fillna("")
y_test = test_df["label"]

# Evaluate the selected baseline model
test_predictions = baseline_model.predict(X_test)

test_df = test_df.copy()
test_df["predicted"] = test_predictions
test_df["correct"] = (
    test_df["label"] == test_df["predicted"]
)

print("===== FINAL TEST RESULTS =====")

print("Accuracy:", round(
    accuracy_score(y_test, test_predictions), 4
))

print("Macro F1:", round(
    f1_score(
        y_test, test_predictions,
        average="macro", zero_division=0
    ), 4
))

print("\nClassification Report:")
print(classification_report(
    y_test, test_predictions,
    zero_division=0
))

# Evaluate each language separately
print("\n===== LANGUAGE COMPARISON =====")

language_results = []

for lang in ["ar", "en"]:
    subset = test_df[test_df["language"] == lang]

    acc = accuracy_score(
        subset["label"], subset["predicted"]
    )
    f1 = f1_score(
        subset["label"], subset["predicted"],
        average="macro", zero_division=0
    )

    language_results.append({
        "Language": lang,
        "Reviews": len(subset),
        "Accuracy": round(acc, 4),
        "Macro F1": round(f1, 4)
    })

language_df = pd.DataFrame(language_results)
display(language_df)

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    labels=["negative", "neutral", "positive"],
    cmap="Blues",
    ax=ax
)

plt.title("Final Test Confusion Matrix")
plt.tight_layout()
plt.savefig(
    results_dir / "confusion_matrix.png",
    dpi=150
)
plt.show()

# Save evaluation results
test_df.to_csv(
    results_dir / "test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

language_df.to_csv(
    results_dir / "language_evaluation.csv",
    index=False
)

print("\n✅ Final evaluation completed and saved!")


In [ ]:

# Step 16: Error Analysis — 10 Failure Cases

import pandas as pd

# Extract incorrectly classified reviews
errors_df = test_df[
    test_df["label"] != test_df["predicted"]
].copy()

print("Total incorrect predictions:", len(errors_df))

# Initial error categories based on prediction direction
def categorize_error(row):
    true_label = row["label"]
    predicted = row["predicted"]

    if true_label == "positive" and predicted == "negative":
        return "Positive misclassified as Negative"

    if true_label == "negative" and predicted == "positive":
        return "Negative misclassified as Positive"

    if true_label == "neutral":
        return "Neutral sentiment misclassification"

    if predicted == "neutral":
        return "Incorrect Neutral prediction"

    return "Other sentiment confusion"

errors_df["error_category"] = errors_df.apply(
    categorize_error, axis=1
)

# Select a reproducible sample of 10 errors
sample_errors = errors_df.sample(
    n=min(10, len(errors_df)),
    random_state=42
).copy()

sample_errors["text_preview"] = (
    sample_errors["text"].astype(str).str[:250]
)

print("\n===== ERROR ANALYSIS SAMPLE =====")

display(
    sample_errors[
        [
            "language",
            "label",
            "predicted",
            "error_category",
            "text_preview"
        ]
    ]
)

print("\n===== ERROR CATEGORY COUNTS =====")
print(errors_df["error_category"].value_counts())

# Save all errors and sample
errors_df.to_csv(
    results_dir / "all_prediction_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_errors.to_csv(
    results_dir / "error_analysis_10.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ Error cases extracted and saved!")


In [ ]:

# Step 17: Review the 10 failure cases in full

# Preserve the selected sample and original row indices
review_errors = sample_errors.copy()

print("===== FULL FAILURE CASES =====")

for case_number, (row_index, row) in enumerate(
    review_errors.iterrows(), start=1
):
    print("\n" + "=" * 60)
    print(f"CASE {case_number} | Original index: {row_index}")
    print("Language:", row["language"])
    print("True Label:", row["label"])
    print("Predicted Label:", row["predicted"])
    print("Error Type:", row["error_category"])
    print("\nFULL REVIEW:")
    print(row["text"])

print("\n✅ All 10 failure cases displayed!")


In [ ]:

# Step 18: Save 10 categorized failure analyses

import pandas as pd

# Map each original test-row index to its analysis
failure_analysis = {
    11: ("Mixed Sentiment",
         "The review praises the research but criticizes missing details."),
    31: ("Non-review Text",
         "This is a general quotation rather than customer feedback."),
    2: ("Long Review and Negation",
        "A strongly positive review contains negative wording and negation."),
    35: ("Negation and Technical Complaint",
         "Repeated sync failures and negation express dissatisfaction."),
    14: ("Dialectal Arabic",
         "Positive sentiment is expressed using colloquial Saudi Arabic."),
    12: ("Dialectal Arabic",
         "Colloquial expressions of praise may be difficult to recognize."),
    20: ("Label Ambiguity",
         "The text sounds mildly positive despite its neutral ground-truth label."),
    3: ("Unusual Wording",
        "The short positive statement has an unusual English structure."),
    43: ("Long Review with Mixed Content",
         "Descriptive passages appear alongside explicit criticism."),
    7: ("Long Review and Domain Vocabulary",
        "Academic terminology dominates a review with positive closing remarks.")
}

# Preserve original test indices and create final annotations
final_errors = sample_errors.copy()
final_errors["test_row_index"] = final_errors.index

final_errors["failure_reason"] = final_errors[
    "test_row_index"
].map(lambda i: failure_analysis[i][0])

final_errors["explanation"] = final_errors[
    "test_row_index"
].map(lambda i: failure_analysis[i][1])

# Keep the important columns
final_errors = final_errors[
    [
        "test_row_index",
        "text",
        "language",
        "label",
        "predicted",
        "error_category",
        "failure_reason",
        "explanation"
    ]
].sort_values("test_row_index")

# Save analysis to Google Drive
output_file = results_dir / "error_analysis_detailed.csv"

final_errors.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("===== FINAL ERROR ANALYSIS =====")

display(
    final_errors[
        [
            "test_row_index",
            "language",
            "label",
            "predicted",
            "failure_reason"
        ]
    ]
)

print("\nTotal analyzed failures:", len(final_errors))
print("Saved to:", output_file)

assert len(final_errors) == 10
assert final_errors["failure_reason"].notna().all()

print("\n✅ Detailed error analysis saved successfully!")


In [ ]:

# Step 19: Save the trained sentiment model

import joblib
from pathlib import Path

# Create models directory in Google Drive
models_dir = project_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Save the selected Word TF-IDF model
model_path = models_dir / "sentiment_classifier.joblib"

joblib.dump(baseline_model, model_path)

print("Model saved to:", model_path)

# Load the model again to verify the saved file
loaded_model = joblib.load(model_path)

# Test model loading with two example reviews
example_reviews = [
    "الخدمة ممتازة والتعامل رائع",
    "The delivery was late and the product was damaged"
]

predictions = loaded_model.predict(example_reviews)

print("\n===== MODEL LOADING TEST =====")

for review, prediction in zip(example_reviews, predictions):
    print("Review:", review)
    print("Prediction:", prediction)
    print("-" * 50)

# Check saved file
assert model_path.exists()
assert len(predictions) == 2

print("\n✅ Model saved and loaded successfully!")


In [ ]:

# Step 20: Build Gradio Demo

import gradio as gr
import pandas as pd
import numpy as np
import joblib

# Load our trained sentiment model
demo_model = joblib.load(
    models_dir / "sentiment_classifier.joblib"
)

# Load previously generated insights
demo_insights = pd.read_csv(
    results_dir / "customer_insights.csv"
)

def analyze_sentiment(text):
    if not text or not text.strip():
        return "Please enter a customer review.", {}

    # Predict sentiment
    prediction = demo_model.predict([text])[0]

    # Class probabilities
    probabilities = demo_model.predict_proba([text])[0]

    scores = {
        str(label): float(prob)
        for label, prob in zip(
            demo_model.classes_,
            probabilities
        )
    }

    return prediction.capitalize(), scores


def show_insights():
    columns = [
        "Cluster",
        "Language",
        "Total Reviews",
        "Positive",
        "Neutral",
        "Negative",
        "Negative Rate (%)"
    ]

    return demo_insights[columns]


with gr.Blocks(
    title="Smart Customer Feedback Analyzer"
) as demo:

    gr.Markdown(
        """
        # Smart Customer Feedback Analyzer

        ### Applied NLP Engineering — Capstone Project

        A bilingual NLP prototype for analyzing
        customer feedback in Arabic and English.

        **Features:** Sentiment Classification,
        Semantic Clustering, and Customer Insights.

        **Note:** Sentiment predictions are experimental.
        Final test accuracy: 50%.
        """
    )

    with gr.Tab("Sentiment Analysis"):

        review_input = gr.Textbox(
            label="Customer Review",
            placeholder="Enter Arabic or English feedback...",
            lines=4
        )

        analyze_button = gr.Button(
            "Analyze Sentiment",
            variant="primary"
        )

        sentiment_output = gr.Textbox(
            label="Predicted Sentiment"
        )

        scores_output = gr.Label(
            label="Model Probability Scores",
            num_top_classes=3
        )

        analyze_button.click(
            fn=analyze_sentiment,
            inputs=review_input,
            outputs=[
                sentiment_output,
                scores_output
            ]
        )

        gr.Examples(
            examples=[
                ["الخدمة ممتازة والتعامل رائع"],
                ["التوصيل تأخر والمنتج وصل تالف"],
                ["The product is excellent."],
                ["The customer service was terrible."]
            ],
            inputs=review_input
        )

    with gr.Tab("Customer Insights"):

        gr.Markdown(
            """
            ## Semantic Clustering Results

            These insights were calculated from
            the prepared training dataset.

            The table displays dataset labels,
            not live model predictions.

            Cluster IDs are exploratory and do not
            represent validated issue categories.
            """
        )

        insights_button = gr.Button(
            "Show Customer Insights"
        )

        insights_output = gr.Dataframe(
            label="Customer Feedback Clusters",
            interactive=False
        )

        insights_button.click(
            fn=show_insights,
            inputs=[],
            outputs=insights_output
        )

# Launch the demo inside Colab
demo.launch(
    share=False,
    debug=True
)


In [ ]:

# Step 21: Verify saved project artifacts

from pathlib import Path
import joblib
import pandas as pd

PROJECT_PATH = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)

MODEL_FILE = PROJECT_PATH / "models" / "sentiment_classifier.joblib"
INSIGHTS_FILE = PROJECT_PATH / "results" / "customer_insights.csv"

required_files = [
    PROJECT_PATH / "data" / "customer_train.csv",
    PROJECT_PATH / "data" / "customer_validation.csv",
    PROJECT_PATH / "data" / "customer_test.csv",
    MODEL_FILE,
    INSIGHTS_FILE,
    PROJECT_PATH / "results" / "confusion_matrix.png",
    PROJECT_PATH / "results" / "language_evaluation.csv",
    PROJECT_PATH / "results" / "error_analysis_detailed.csv"
]

print("===== PROJECT FILE CHECK =====")

for path in required_files:
    print(
        "✅" if path.exists() else "❌",
        path.relative_to(PROJECT_PATH)
    )

assert all(path.exists() for path in required_files)

# Reload the model independently
check_model = joblib.load(MODEL_FILE)

# Reload saved insights independently
check_insights = pd.read_csv(INSIGHTS_FILE)

# Test prediction after loading
prediction = check_model.predict([
    "The customer service was terrible."
])[0]

print("\n===== RELOAD TEST =====")
print("Model prediction:", prediction)
print("Insights rows:", len(check_insights))

assert prediction in ["positive", "negative", "neutral"]
assert len(check_insights) == 4

print("\n✅ All required artifacts verified successfully!")


In [ ]:

# Restore training and validation data
# No need to retrain previous models

import pandas as pd
from pathlib import Path

# Project folder in Google Drive
project_dir = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)

# Connect Drive if needed
if not (project_dir / "data").exists():
    from google.colab import drive
    drive.mount("/content/drive")

data_dir = project_dir / "data"

# Load saved datasets
train_df = pd.read_csv(data_dir / "customer_train.csv")
val_df = pd.read_csv(data_dir / "customer_validation.csv")

# Restore variables
X_train = train_df["text"].fillna("")
y_train = train_df["label"]

X_val = val_df["text"].fillna("")
y_val = val_df["label"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

assert len(X_train) == 150
assert len(X_val) == 48

print("✅ Training variables restored successfully!")


In [ ]:

# Step 22: Word + Character TF-IDF Optimization
# Validation-only experiment

import pandas as pd

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

baseline_f1 = 0.491527
experiments = []
candidate_models = {}

for c in [0.1, 1.0, 10.0, 100.0]:

    for char_weight in [0.5, 1.0, 2.0]:

        features = FeatureUnion([
            (
                "word",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    max_features=10000,
                    sublinear_tf=True
                )
            ),
            (
                "char",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(2, 5),
                    max_features=20000,
                    sublinear_tf=True
                )
            )
        ],
        transformer_weights={
            "word": 1.0,
            "char": char_weight
        })

        model = Pipeline([
            ("features", features),
            (
                "classifier",
                LogisticRegression(
                    C=c,
                    max_iter=2000,
                    random_state=42
                )
            )
        ])

        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        f1 = f1_score(
            y_val,
            predictions,
            average="macro",
            zero_division=0
        )

        acc = accuracy_score(y_val, predictions)

        key = (c, char_weight)
        candidate_models[key] = model

        experiments.append({
            "C": c,
            "Char Weight": char_weight,
            "Accuracy": acc,
            "Macro F1": f1
        })

results_df = pd.DataFrame(experiments)

results_df = results_df.sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

print("===== OPTIMIZATION RESULTS =====")
display(results_df)

best = results_df.iloc[0]

best_key = (
    float(best["C"]),
    float(best["Char Weight"])
)

optimized_model = candidate_models[best_key]

print("\n===== BEST CONFIGURATION =====")
print("C:", best["C"])
print("Char Weight:", best["Char Weight"])
print("Validation Accuracy:", round(best["Accuracy"], 4))
print("Validation Macro F1:", round(best["Macro F1"], 4))

print("\n===== BASELINE COMPARISON =====")
print("Baseline F1:", round(baseline_f1, 4))
print("Optimized F1:", round(best["Macro F1"], 4))

if best["Macro F1"] > baseline_f1:
    print("✅ Improvement on validation!")
else:
    print("⚠️ Baseline remains better on validation.")

print("\n✅ Optimization experiment completed!")


In [ ]:

# Step 23: Baseline validation performance by language

import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)

# Mount Drive if not already mounted
if not (PROJECT_DIR / "data").exists():
    from google.colab import drive
    drive.mount("/content/drive")

val_data = pd.read_csv(
    PROJECT_DIR / "data" / "customer_validation.csv"
)

saved_model = joblib.load(
    PROJECT_DIR / "models" / "sentiment_classifier.joblib"
)

val_data["baseline_prediction"] = saved_model.predict(
    val_data["text"].fillna("")
)

results = []

for lang in ["ar", "en"]:
    subset = val_data[val_data["language"] == lang]

    results.append({
        "Language": "Arabic" if lang == "ar" else "English",
        "Reviews": len(subset),
        "Accuracy": accuracy_score(
            subset["label"],
            subset["baseline_prediction"]
        ),
        "Macro F1": f1_score(
            subset["label"],
            subset["baseline_prediction"],
            average="macro",
            zero_division=0
        )
    })

slice_results = pd.DataFrame(results)

print("===== BASELINE VALIDATION BY LANGUAGE =====")
display(slice_results)

# Save comparison
results_dir = PROJECT_DIR / "results"
results_dir.mkdir(exist_ok=True)

slice_results.to_csv(
    results_dir / "validation_by_language.csv",
    index=False
)

print("\n✅ Language comparison saved successfully!")


In [ ]:

# Step 24: Compare Baseline vs Embeddings by Language
# Uses validation data only

import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Load saved datasets
train_data = pd.read_csv(
    PROJECT_DIR / "data" / "customer_train.csv"
)

val_data = pd.read_csv(
    PROJECT_DIR / "data" / "customer_validation.csv"
)

# Load multilingual embedding model
embedding_name = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_encoder = SentenceTransformer(
    embedding_name,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Encode train and validation texts
train_vectors = embedding_encoder.encode(
    train_data["text"].fillna("").tolist(),
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

val_vectors = embedding_encoder.encode(
    val_data["text"].fillna("").tolist(),
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Train embedding-based classifier
embedding_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

embedding_classifier.fit(
    train_vectors,
    train_data["label"]
)

# Get predictions from both models
baseline_preds = saved_model.predict(
    val_data["text"].fillna("")
)

embedding_preds = embedding_classifier.predict(
    val_vectors
)

# Compare performance by language
rows = []

for lang in ["ar", "en"]:
    mask = val_data["language"] == lang
    true_labels = val_data.loc[mask, "label"]

    for model_name, preds in [
        ("Baseline TF-IDF", baseline_preds),
        ("Multilingual Embeddings", embedding_preds)
    ]:
        selected_preds = preds[mask.to_numpy()]

        rows.append({
            "Language": "Arabic" if lang == "ar" else "English",
            "Model": model_name,
            "Reviews": int(mask.sum()),
            "Accuracy": round(
                accuracy_score(true_labels, selected_preds), 4
            ),
            "Macro F1": round(
                f1_score(
                    true_labels,
                    selected_preds,
                    average="macro",
                    zero_division=0
                ), 4
            )
        })

comparison_by_language = pd.DataFrame(rows)

print("===== BEFORE / AFTER BY LANGUAGE =====")
display(comparison_by_language)

# Save for GitHub and README
comparison_by_language.to_csv(
    results_dir / "model_comparison_by_language.csv",
    index=False
)

print("\n✅ Before/After comparison saved successfully!")


In [ ]:

# Step 25: Generate an evidence-based insights conclusion

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)

results_dir = PROJECT_DIR / "results"

insights = pd.read_csv(
    results_dir / "customer_insights.csv"
)

# Identify clusters with highest and lowest negative rates
highest = insights.loc[
    insights["Negative Rate (%)"].idxmax()
]

lowest = insights.loc[
    insights["Negative Rate (%)"].idxmin()
]

conclusion = (
    f"Cluster {highest['Cluster']} has the highest observed "
    f"negative review rate ({highest['Negative Rate (%)']:.1f}%), "
    f"while cluster {lowest['Cluster']} has the lowest "
    f"({lowest['Negative Rate (%)']:.1f}%). "
    "These results are based on labeled training reviews. "
    "Clusters are exploratory and require further validation "
    "before they can be treated as specific customer issues."
)

print("===== CUSTOMER INSIGHTS CONCLUSION =====")
print(conclusion)

# Save conclusion for the final app and README
conclusion_file = results_dir / "insights_conclusion.txt"

conclusion_file.write_text(
    conclusion,
    encoding="utf-8"
)

print("\n✅ Insights conclusion generated and saved!")


In [ ]:

# Step 26: Final Gradio Interface
# Sentiment Analysis + Uncertainty Warning + Customer Insights

import gradio as gr
import pandas as pd
import joblib
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)

MODEL_PATH = PROJECT_DIR / "models" / "sentiment_classifier.joblib"
RESULTS_DIR = PROJECT_DIR / "results"

# Load saved project artifacts
final_model = joblib.load(MODEL_PATH)
final_insights = pd.read_csv(
    RESULTS_DIR / "customer_insights.csv"
)

final_conclusion = (
    RESULTS_DIR / "insights_conclusion.txt"
).read_text(encoding="utf-8")


def predict_feedback(text):

    if not text or not text.strip():
        return "Please enter a review.", {}, "No input provided."

    # Predict sentiment
    prediction = final_model.predict([text])[0]

    # Get model probabilities
    probabilities = final_model.predict_proba([text])[0]

    scores = {
        str(label): float(prob)
        for label, prob in zip(
            final_model.classes_,
            probabilities
        )
    }

    # Check how close the top two probabilities are
    sorted_probs = sorted(probabilities, reverse=True)
    margin = float(sorted_probs[0] - sorted_probs[1])

    # Heuristic uncertainty indicator
    if margin < 0.10:
        warning = (
            "⚠️ Uncertain prediction: the top two model "
            "probability scores are close. Interpret with caution."
        )
    else:
        warning = (
            "Prediction available. Probability scores are "
            "model estimates, not guaranteed confidence."
        )

    return prediction.capitalize(), scores, warning


def get_customer_insights():

    columns = [
        "Cluster",
        "Language",
        "Total Reviews",
        "Positive",
        "Neutral",
        "Negative",
        "Negative Rate (%)"
    ]

    return final_insights[columns], final_conclusion


with gr.Blocks(
    title="Smart Customer Feedback Analyzer"
) as final_demo:

    gr.Markdown(
        """
        # Smart Customer Feedback Analyzer

        **Applied NLP Engineering — Capstone Project**

        A bilingual NLP prototype for analyzing
        Arabic and English customer feedback.

        Features:
        - Sentiment Classification
        - Model Probability Scores
        - Uncertainty Indicator
        - Semantic Clustering Insights

        **Final Test Accuracy: 50% | Macro F1: 48.48%**

        This is an experimental prototype.
        """
    )

    with gr.Tab("Sentiment Analysis"):

        review_input = gr.Textbox(
            label="Customer Review",
            placeholder="Enter Arabic or English feedback...",
            lines=4
        )

        analyze_button = gr.Button(
            "Analyze Sentiment",
            variant="primary"
        )

        sentiment_output = gr.Textbox(
            label="Predicted Sentiment"
        )

        scores_output = gr.Label(
            label="Model Probability Scores",
            num_top_classes=3
        )

        warning_output = gr.Textbox(
            label="Prediction Reliability Note",
            interactive=False
        )

        analyze_button.click(
            fn=predict_feedback,
            inputs=review_input,
            outputs=[
                sentiment_output,
                scores_output,
                warning_output
            ]
        )

        gr.Examples(
            examples=[
                ["الخدمة ممتازة والتعامل رائع"],
                ["التوصيل تأخر والمنتج وصل تالف"],
                ["The product is excellent."],
                ["The customer service was terrible."]
            ],
            inputs=review_input
        )

    with gr.Tab("Customer Insights"):

        gr.Markdown(
            """
            ## Customer Feedback Insights

            Exploratory semantic clusters derived
            from labeled training data.

            The negative rates are based on original
            dataset labels, not live predictions.

            Cluster IDs are not validated issue categories.
            """
        )

        insights_button = gr.Button(
            "Show Customer Insights"
        )

        insights_table = gr.Dataframe(
            label="Customer Feedback Clusters",
            interactive=False
        )

        conclusion_output = gr.Textbox(
            label="Evidence-Based Insights Summary",
            lines=4,
            interactive=False
        )

        insights_button.click(
            fn=get_customer_insights,
            inputs=[],
            outputs=[
                insights_table,
                conclusion_output
            ]
        )

final_demo.launch(
    share=False,
    debug=True
)


In [ ]:

# Step 27: Test Generative LLM
# Qwen2.5-1.5B-Instruct with Hugging Face Transformers

!pip -q install -U transformers accelerate

import torch
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

# Project files
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)

# Mount Google Drive if needed
if not (PROJECT_DIR / "results").exists():
    from google.colab import drive
    drive.mount("/content/drive")

RESULTS_DIR = PROJECT_DIR / "results"

# Read real clustering results
insights = pd.read_csv(
    RESULTS_DIR / "customer_insights.csv"
)

# Check GPU
print("GPU available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Please enable T4 GPU before loading the LLM."
    )

# Model name
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

print("Loading Generative LLM...")
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

# Prepare structured evidence from actual data
columns = [
    "Cluster",
    "Language",
    "Total Reviews",
    "Positive",
    "Neutral",
    "Negative",
    "Negative Rate (%)"
]

evidence = insights[columns].to_string(index=False)

# Grounded prompt
messages = [
    {
        "role": "system",
        "content": (
            "You are a customer feedback analyst. "
            "Use only the provided statistics. "
            "Do not invent numbers, customer issues, "
            "or causes. Cluster IDs are exploratory "
            "and are not validated issue categories. "
            "Write a short, factual English summary."
        )
    },
    {
        "role": "user",
        "content": (
            "Summarize these customer feedback results "
            "in 3 short sentences.\n\n"
            f"{evidence}\n\n"
            "Mention the cluster with the highest "
            "negative rate and the cluster with "
            "the lowest negative rate. "
            "State that these are exploratory results "
            "based on labeled training reviews."
        )
    }
]

# Format prompt for Qwen
# Fix: explicitly create input_ids and attention_mask

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
)

model_inputs = {
    key: value.to(llm_model.device)
    for key, value in model_inputs.items()
}

print("\nGenerating customer insights...")

with torch.inference_mode():
    output_ids = llm_model.generate(
        **model_inputs,
        max_new_tokens=170,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only newly generated tokens
input_length = model_inputs["input_ids"].shape[1]

generated_ids = output_ids[0][input_length:]

llm_summary = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

print("\n===== LLM GENERATED INSIGHTS =====")
print(llm_summary)

print("\n✅ Generative LLM test completed!")


In [ ]:

# Step 28: Grounded LLM Summary + Validation

import re
import torch
import pandas as pd

# Load real statistics
insights = pd.read_csv(
    RESULTS_DIR / "customer_insights.csv"
)

# Calculate facts using Python
highest = insights.loc[
    insights["Negative Rate (%)"].idxmax()
]
lowest = insights.loc[
    insights["Negative Rate (%)"].idxmin()
]

high_id = str(highest["Cluster"])
low_id = str(lowest["Cluster"])
high_rate = f'{highest["Negative Rate (%)"]:.1f}%'
low_rate = f'{lowest["Negative Rate (%)"]:.1f}%'

verified_summary = (
    f"Cluster {high_id} has the highest negative review "
    f"rate ({high_rate}), while cluster {low_id} has the "
    f"lowest ({low_rate}). These statistics are based "
    "on labeled training reviews. Clusters are "
    "exploratory and are not validated issue categories."
)

print("===== PYTHON VERIFIED FACTS =====")
print(verified_summary)

# Give Qwen only the verified facts
messages = [
    {
        "role": "system",
        "content": (
            "You are a careful NLP reporting assistant. "
            "Rewrite the supplied facts in clear English. "
            "Preserve all cluster IDs, numbers, and their "
            "highest/lowest relationships exactly. "
            "Do not add new facts or percentages."
        )
    },
    {
        "role": "user",
        "content": (
            "Rewrite these verified facts in 2-3 short "
            "sentences. Do not change any values:\n\n"
            + verified_summary
        )
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer(
    prompt, return_tensors="pt"
)

model_inputs = {
    key: value.to(llm_model.device)
    for key, value in model_inputs.items()
}

with torch.inference_mode():
    output = llm_model.generate(
        **model_inputs,
        max_new_tokens=160,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated = tokenizer.decode(
    output[0][model_inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

# Basic validation:
# Ensure the model does not change cluster IDs or percentages.
generated_ids = set(
    re.findall(r'\b(?:ar|en)_\d+\b', generated)
)

generated_rates = set(
    re.findall(r'\d+(?:\.\d+)?%', generated)
)

expected_ids = {high_id, low_id}
expected_rates = {high_rate, low_rate}

facts_match = (
    generated_ids == expected_ids
    and generated_rates == expected_rates
    and re.search(
        rf'\b{re.escape(high_id)}\b[^.!?\n]*'
        rf'{re.escape(high_rate)}',
        generated
    ) is not None
    and re.search(
        rf'\b{re.escape(low_id)}\b[^.!?\n]*'
        rf'{re.escape(low_rate)}',
        generated
    ) is not None
)

# Additional guard against reversed highest/lowest claims
relationship_safe = (
    "highest" in generated.lower()
    and "lowest" in generated.lower()
    and generated.find(high_id) < generated.find(low_id)
    and generated.lower().find("highest")
        < generated.find(high_id)
    and generated.lower().find("lowest")
        < generated.find(low_id)
)

llm_valid = bool(facts_match and relationship_safe)

if llm_valid:
    final_summary = generated
    print("\n✅ LLM summary passed basic validation.")
else:
    final_summary = verified_summary
    print("\n⚠️ LLM output did not pass validation.")
    print("Using the Python-verified summary instead.")

# Save outputs
(RESULTS_DIR / "llm_raw_summary.txt").write_text(
    generated, encoding="utf-8"
)

(RESULTS_DIR / "llm_verified_summary.txt").write_text(
    final_summary, encoding="utf-8"
)

print("\n===== RAW LLM SUMMARY =====")
print(generated)

print("\n===== FINAL VERIFIED SUMMARY =====")
print(final_summary)

print("\n✅ Step 28 completed and saved!")


In [ ]:

# Step 29: Improve LLM output validation
# Use the previously saved generated summary

import re
import pandas as pd

insights = pd.read_csv(
    RESULTS_DIR / "customer_insights.csv"
)

raw_summary = (
    RESULTS_DIR / "llm_raw_summary.txt"
).read_text(encoding="utf-8")

highest = insights.loc[
    insights["Negative Rate (%)"].idxmax()
]
lowest = insights.loc[
    insights["Negative Rate (%)"].idxmin()
]

high_id = str(highest["Cluster"])
low_id = str(lowest["Cluster"])
high_rate = f'{highest["Negative Rate (%)"]:.1f}'
low_rate = f'{lowest["Negative Rate (%)"]:.1f}'

# Accept both "at 56.2%" and "(56.2%)"
def contains_cluster_rate(text, cluster, rate):
    pattern = (
        rf'\b{re.escape(cluster)}\b'
        rf'[^.!?\n]{{0,100}}'
        rf'\b{re.escape(rate)}%'
    )
    return bool(re.search(pattern, text, re.I))

numeric_valid = (
    contains_cluster_rate(raw_summary, high_id, high_rate)
    and contains_cluster_rate(raw_summary, low_id, low_rate)
)

# Check that no unexpected cluster IDs or rates appear
mentioned_clusters = set(re.findall(
    r'\b(?:ar|en)_\d+\b', raw_summary
))
mentioned_rates = set(re.findall(
    r'\b\d+(?:\.\d+)?%', raw_summary
))

numeric_valid = numeric_valid and (
    mentioned_clusters == {high_id, low_id}
    and mentioned_rates == {
        high_rate + "%",
        low_rate + "%"
    }
)

# Simple screening for unsupported interpretations
unsupported_phrases = [
    "most dissatisfied customers",
    "least dissatisfied customers",
    "customer satisfaction improved",
    "caused by",
    "proves that"
]

unsupported_found = [
    phrase for phrase in unsupported_phrases
    if phrase in raw_summary.lower()
]

# Deterministic fallback
verified_summary = (
    f"Cluster {high_id} has the highest observed "
    f"negative review rate ({high_rate}%), while "
    f"cluster {low_id} has the lowest ({low_rate}%). "
    "These statistics are based on labeled training "
    "reviews. The clusters are exploratory and are "
    "not validated customer issue categories."
)

if numeric_valid and not unsupported_found:
    final_summary = raw_summary
    summary_source = "LLM generated - basic checks passed"
else:
    final_summary = verified_summary
    summary_source = "Python verified fallback"

# Save the final result
(RESULTS_DIR / "llm_verified_summary.txt").write_text(
    final_summary, encoding="utf-8"
)

print("===== VALIDATION RESULTS =====")
print("Numeric facts valid:", numeric_valid)
print("Unsupported phrases:", unsupported_found)
print("Summary source:", summary_source)

print("\n===== FINAL SUMMARY =====")
print(final_summary)

print("\n✅ Step 29 completed!")


In [ ]:

# Step 30: Gradio App with Live Generative LLM
# Sentiment + Clustering Insights + Qwen Summary

import gradio as gr
import pandas as pd
import joblib
import torch
import re
from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)
RESULTS = PROJECT / "results"

sentiment_model = joblib.load(
    PROJECT / "models/sentiment_classifier.joblib"
)
insights_data = pd.read_csv(
    RESULTS / "customer_insights.csv"
)

# Check the LLM is already loaded
assert "llm_model" in globals(), "Run Step 27 first."
assert "tokenizer" in globals(), "Run Step 27 first."


def sentiment_analysis(text):
    if not text or not text.strip():
        return "", {}, "Please enter a review."

    prediction = sentiment_model.predict([text])[0]
    probabilities = sentiment_model.predict_proba([text])[0]

    scores = {
        str(label): float(prob)
        for label, prob in zip(
            sentiment_model.classes_, probabilities
        )
    }

    top_two = sorted(probabilities, reverse=True)[:2]
    margin = float(top_two[0] - top_two[1])

    if margin < 0.10:
        note = (
            "Uncertain prediction: the top two probability "
            "scores are close. Interpret with caution."
        )
    else:
        note = (
            "Probability scores are model estimates, "
            "not guaranteed confidence."
        )

    return str(prediction).capitalize(), scores, note


def generate_llm_insights():
    # Calculate facts using Python
    highest = insights_data.loc[
        insights_data["Negative Rate (%)"].idxmax()
    ]
    lowest = insights_data.loc[
        insights_data["Negative Rate (%)"].idxmin()
    ]

    high_id = str(highest["Cluster"])
    low_id = str(lowest["Cluster"])
    high_rate = f'{highest["Negative Rate (%)"]:.1f}'
    low_rate = f'{lowest["Negative Rate (%)"]:.1f}'

    fallback = (
        f"Cluster {high_id} has the highest observed "
        f"negative review rate ({high_rate}%), while "
        f"cluster {low_id} has the lowest ({low_rate}%). "
        "These results are based on labeled training reviews. "
        "The clusters are exploratory, not validated "
        "customer issue categories."
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are an NLP reporting assistant. "
                "Rewrite only the verified facts. "
                "Preserve all IDs and percentages. "
                "Do not infer customer satisfaction, "
                "causes, trends, or specific issues. "
                "Keep the summary under 80 words."
            )
        },
        {
            "role": "user",
            "content": fallback
        }
    ]

    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt, return_tensors="pt"
        ).to(llm_model.device)

        with torch.inference_mode():
            generated_ids = llm_model.generate(
                **inputs,
                max_new_tokens=130,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        raw = tokenizer.decode(
            generated_ids[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        # Basic numerical and wording checks
        def has_pair(cluster, rate):
            pattern = (
                rf'\b{re.escape(cluster)}\b'
                rf'[^.!?\n]{{0,100}}'
                rf'\b{re.escape(rate)}%'
            )
            return bool(re.search(pattern, raw, re.I))

        clusters = set(re.findall(
            r'\b(?:ar|en)_\d+\b', raw
        ))
        rates = set(re.findall(
            r'\b\d+(?:\.\d+)?%', raw
        ))

        banned = [
            "most dissatisfied customers",
            "least dissatisfied customers",
            "caused by",
            "proves that",
            "customer satisfaction improved"
        ]

        numeric_ok = (
            clusters == {high_id, low_id}
            and rates == {
                high_rate + "%",
                low_rate + "%"
            }
            and has_pair(high_id, high_rate)
            and has_pair(low_id, low_rate)
        )

        text_lower = raw.lower()
        high_pos = text_lower.find("highest")
        low_pos = text_lower.find("lowest")

        relations_ok = (
            high_pos >= 0
            and low_pos >= 0
            and high_pos < raw.find(high_id)
            and low_pos < raw.find(low_id)
        )

        safe = (
            numeric_ok
            and relations_ok
            and not any(p in text_lower for p in banned)
        )

        if safe:
            summary = raw
            status = (
                "LLM generated — basic validation passed. "
                "Human review is still recommended."
            )
        else:
            summary = fallback
            status = (
                "LLM generated, but validation rejected "
                "the output. Python fallback displayed."
            )

        (RESULTS / "llm_latest_raw.txt").write_text(
            raw, encoding="utf-8"
        )

    except Exception as error:
        summary = fallback
        status = (
            "LLM unavailable; verified fallback used. "
            f"Error type: {type(error).__name__}"
        )

    return summary, status


def show_customer_insights():
    cols = [
        "Cluster", "Language", "Total Reviews",
        "Positive", "Neutral", "Negative",
        "Negative Rate (%)"
    ]
    return insights_data[cols]


with gr.Blocks(
    title="Smart Customer Feedback Analyzer"
) as demo_llm:

    gr.Markdown("""
    # Smart Customer Feedback Analyzer
    ### SDAIA Academy — Applied NLP Engineering

    Bilingual NLP prototype using:
    - TF-IDF + Logistic Regression for sentiment classification
    - Multilingual Sentence Embeddings + K-Means for clustering
    - Qwen2.5-1.5B-Instruct for grounded insight generation

    **Test Accuracy: 50% | Macro F1: 48.48%**

    Experimental prototype. Results are not production-grade.
    """)

    with gr.Tab("Sentiment Analysis"):
        review = gr.Textbox(
            label="Customer Review (Arabic / English)",
            lines=4
        )

        predict_button = gr.Button(
            "Analyze Sentiment", variant="primary"
        )

        sentiment = gr.Textbox(
            label="Predicted Sentiment"
        )
        probabilities = gr.Label(
            label="Model Probability Scores",
            num_top_classes=3
        )
        reliability = gr.Textbox(
            label="Prediction Reliability Note"
        )

        predict_button.click(
            sentiment_analysis,
            inputs=review,
            outputs=[sentiment, probabilities, reliability]
        )

    with gr.Tab("Customer Insights + LLM"):

        gr.Markdown("""
        ## Semantic Clustering Results

        Statistics are based on original labels in the
        training dataset, not live sentiment predictions.
        Clusters are exploratory, not validated issue types.
        """)

        show_button = gr.Button("Show Customer Insights")
        insights_table = gr.Dataframe(
            label="Cluster Statistics",
            interactive=False
        )

        show_button.click(
            show_customer_insights,
            inputs=[],
            outputs=insights_table
        )

        generate_button = gr.Button(
            "Generate LLM Summary",
            variant="primary"
        )

        summary_output = gr.Textbox(
            label="Generated / Verified Summary",
            lines=5
        )

        status_output = gr.Textbox(
            label="LLM Validation Status"
        )

        generate_button.click(
            generate_llm_insights,
            inputs=[],
            outputs=[summary_output, status_output]
        )

demo_llm.launch(share=False, debug=False)


In [ ]:

# Step 31: Create standalone application files

from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)

PROJECT.mkdir(parents=True, exist_ok=True)

app_code = r'''
import re
from pathlib import Path

import gradio as gr
import joblib
import pandas as pd

ROOT = Path(__file__).resolve().parent

model = joblib.load(
    ROOT / "models" / "sentiment_classifier.joblib"
)

insights = pd.read_csv(
    ROOT / "results" / "customer_insights.csv"
)

_llm = None
_tokenizer = None


def predict_review(text):
    if not text or not text.strip():
        return "", {}, "Please enter a review."

    prediction = model.predict([text])[0]
    probs = model.predict_proba([text])[0]

    scores = {
        str(label): float(value)
        for label, value in zip(model.classes_, probs)
    }

    top = sorted(probs, reverse=True)
    margin = float(top[0] - top[1])

    note = (
        "Uncertain prediction: probability scores "
        "are close. Interpret with caution."
        if margin < 0.10 else
        "Model probability estimates are not "
        "guaranteed confidence."
    )

    return str(prediction).capitalize(), scores, note


def show_insights():
    return insights[
        ["Cluster", "Language", "Total Reviews",
         "Positive", "Neutral", "Negative",
         "Negative Rate (%)"]
    ]


def generate_summary():
    global _llm, _tokenizer

    high = insights.loc[
        insights["Negative Rate (%)"].idxmax()
    ]
    low = insights.loc[
        insights["Negative Rate (%)"].idxmin()
    ]

    high_id, low_id = str(high["Cluster"]), str(low["Cluster"])
    high_rate = f'{high["Negative Rate (%)"]:.1f}'
    low_rate = f'{low["Negative Rate (%)"]:.1f}'

    fallback = (
        f"Cluster {high_id} has the highest observed "
        f"negative review rate ({high_rate}%), "
        f"while cluster {low_id} has the lowest "
        f"({low_rate}%). These statistics are based "
        "on labeled training reviews. Clusters are "
        "exploratory, not validated issue categories."
    )

    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        if _llm is None:
            if not torch.cuda.is_available():
                return fallback, (
                    "LLM unavailable: GPU required. "
                    "Python fallback displayed."
                )

            name = "Qwen/Qwen2.5-1.5B-Instruct"
            _tokenizer = AutoTokenizer.from_pretrained(name)
            _llm = AutoModelForCausalLM.from_pretrained(
                name, dtype=torch.float16, device_map="auto"
            )

        messages = [
            {
                "role": "system",
                "content": (
                    "Rewrite verified facts only. "
                    "Keep IDs and numbers exact. "
                    "Do not infer customer satisfaction, "
                    "trends, causes or specific issues."
                )
            },
            {"role": "user", "content": fallback}
        ]

        prompt = _tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=True
        )

        inputs = _tokenizer(
            prompt, return_tensors="pt"
        ).to(_llm.device)

        with torch.inference_mode():
            output = _llm.generate(
                **inputs,
                max_new_tokens=130,
                do_sample=False,
                pad_token_id=_tokenizer.eos_token_id
            )

        raw = _tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        def has_pair(cluster, rate):
            return bool(re.search(
                rf'\b{re.escape(cluster)}\b'
                rf'[^.!?\n]{{0,100}}'
                rf'\b{re.escape(rate)}%',
                raw, re.I
            ))

        found_ids = set(re.findall(
            r'\b(?:ar|en)_\d+\b', raw
        ))
        found_rates = set(re.findall(
            r'\b\d+(?:\.\d+)?%', raw
        ))

        banned = [
            "most dissatisfied customers",
            "least dissatisfied customers",
            "caused by",
            "proves that",
            "customer satisfaction improved"
        ]

        lower = raw.lower()
        numeric_ok = (
            found_ids == {high_id, low_id}
            and found_rates == {
                high_rate + "%", low_rate + "%"
            }
            and has_pair(high_id, high_rate)
            and has_pair(low_id, low_rate)
        )

        relations_ok = (
            0 <= lower.find("highest") < raw.find(high_id)
            and 0 <= lower.find("lowest") < raw.find(low_id)
        )

        if (
            numeric_ok and relations_ok
            and not any(p in lower for p in banned)
        ):
            return raw, (
                "LLM generated — basic checks passed. "
                "Human review recommended."
            )

        return fallback, (
            "LLM generated, but validation rejected "
            "the output. Python fallback displayed."
        )

    except Exception as error:
        return fallback, (
            "LLM unavailable; Python fallback displayed. "
            f"Error type: {type(error).__name__}"
        )


with gr.Blocks(
    title="Smart Customer Feedback Analyzer"
) as demo:

    gr.Markdown("""
    # Smart Customer Feedback Analyzer
    **SDAIA Academy — Applied NLP Engineering**

    Arabic & English customer feedback prototype.

    - TF-IDF + Logistic Regression: sentiment
    - Sentence Transformers + K-Means: offline clustering
    - Qwen2.5-1.5B-Instruct: optional live LLM summary

    **Test Accuracy: 50% | Macro F1: 48.48%**

    Experimental results; not production-grade.
    """)

    with gr.Tab("Sentiment Analysis"):
        review = gr.Textbox(
            label="Customer Review", lines=4
        )
        button = gr.Button(
            "Analyze Sentiment", variant="primary"
        )
        sentiment = gr.Textbox(
            label="Predicted Sentiment"
        )
        scores = gr.Label(
            label="Model Probability Scores"
        )
        note = gr.Textbox(
            label="Prediction Reliability Note"
        )
        button.click(
            predict_review,
            inputs=review,
            outputs=[sentiment, scores, note]
        )

    with gr.Tab("Customer Insights + LLM"):
        gr.Markdown(
            "Exploratory clusters based on original "
            "training labels, not live predictions."
        )

        table = gr.Dataframe(interactive=False)
        show = gr.Button("Show Customer Insights")
        show.click(show_insights, outputs=table)

        generate = gr.Button(
            "Generate LLM Summary", variant="primary"
        )
        summary = gr.Textbox(
            label="Generated / Verified Summary", lines=5
        )
        status = gr.Textbox(
            label="LLM Validation Status"
        )
        generate.click(
            generate_summary,
            outputs=[summary, status]
        )


if __name__ == "__main__":
    demo.launch()
'''

(PROJECT / "app.py").write_text(
    app_code, encoding="utf-8"
)

requirements = """gradio>=5,<7
pandas>=2,<3
numpy>=1.26,<3
scikit-learn>=1.5,<2
joblib>=1.4,<2
torch>=2.3
transformers>=4.51,<5
accelerate>=1.0
sentence-transformers>=3,<6
"""

(PROJECT / "requirements.txt").write_text(
    requirements, encoding="utf-8"
)

print("===== APPLICATION FILES =====")
for filename in ["app.py", "requirements.txt"]:
    file = PROJECT / filename
    print(file.name, "—", file.stat().st_size, "bytes")

# Validate Python syntax without launching the app
import ast
ast.parse((PROJECT / "app.py").read_text(encoding="utf-8"))

print("\n✅ Standalone application files created successfully!")


In [ ]:

# Step 32: Test standalone app.py
# Run as a separate Python process

import os
import sys
import time
import subprocess
import requests
from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)

APP_FILE = PROJECT / "app.py"
LOG_FILE = PROJECT / "app_test.log"

assert APP_FILE.exists(), "app.py not found!"

# Start a separate process (not Notebook variables)
log_handle = open(LOG_FILE, "w", encoding="utf-8")

env = os.environ.copy()
env["GRADIO_SERVER_PORT"] = "7862"
env["GRADIO_SERVER_NAME"] = "127.0.0.1"

app_process = subprocess.Popen(
    [sys.executable, "-u", str(APP_FILE)],
    cwd=str(PROJECT),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env
)

log_handle.close()

print("Starting standalone application...")

app_ready = False

for attempt in range(20):
    if app_process.poll() is not None:
        break

    try:
        response = requests.get(
            "http://127.0.0.1:7862/",
            timeout=2
        )
        if response.status_code == 200:
            app_ready = True
            break
    except requests.exceptions.RequestException:
        pass

    time.sleep(1)

print("\n===== STANDALONE APP TEST =====")

if app_ready:
    print("✅ app.py started successfully!")
    print("✅ Local HTTP server is responding.")
    print("Port: 7862")
else:
    print("❌ App did not become ready.")

print("\n===== APPLICATION LOG =====")
print(
    LOG_FILE.read_text(encoding="utf-8")[-3000:]
)

print("\nProcess running:", app_process.poll() is None)


In [ ]:

# Step 33: Create professional README.md

from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)

readme = """# Smart Customer Feedback Analyzer

### Applied NLP Engineering — Capstone Project

Developed as an individual capstone project for the
AI Engineering Track — Applied NLP Engineering Bootcamp.

**SDAIA Academy:** https://github.com/SDAIAAcademy

## 1. Problem

Organizations receive large amounts of customer feedback
in different languages. Manually reviewing this feedback
can be time-consuming.

This project demonstrates an NLP prototype that:

- Classifies Arabic and English text sentiment.
- Explores semantic clusters using text embeddings.
- Summarizes observed cluster statistics.
- Uses a generative LLM with basic output validation.

## 2. Data

Dataset:
https://huggingface.co/datasets/clapAI/MultiLingualSentiment

The prepared sample contains:

| Split | Arabic | English | Total |
|---|---:|---:|---:|
| Train | 75 | 75 | 150 |
| Validation | 24 | 24 | 48 |
| Test | 24 | 24 | 48 |

Classes: Positive, Neutral, Negative.

The sample is balanced by language and sentiment.

Limitation: The source contains multiple domains,
including customer reviews, book reviews, and social
media posts. It is not exclusively customer feedback.

## 3. Approach

### Sentiment Classification

Baseline:
Word TF-IDF + Logistic Regression.

Alternative experiments:
- Multilingual Sentence Embeddings + Logistic Regression
- Character-level TF-IDF + Logistic Regression
- Combined Word and Character TF-IDF

The model with the highest validation Macro F1
was selected for the final evaluation.

### Semantic Clustering

Model:
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

Sentence embeddings are used with K-Means to explore
similar feedback.

Clustering is performed separately for Arabic and English
on a filtered training-data subset.

The application displays precomputed cluster statistics.
It does not assign new user reviews to clusters live.

### Generative LLM

Model: Qwen/Qwen2.5-1.5B-Instruct

Library: Hugging Face Transformers.

The LLM receives Python-computed cluster statistics
and generates a natural-language summary.

Basic checks examine the generated output.
When validation fails or the LLM is unavailable,
a deterministic Python summary is displayed.

This validation is limited and does not guarantee
that all possible hallucinations will be detected.

### Fine-Tuning

No Transformer fine-tuning was performed.

The sentiment classifier was trained on task-specific data.
The embedding model and the LLM use pretrained weights.

## 4. Results

### Validation Comparison

| Model | Accuracy | Macro F1 |
|---|---:|---:|
| Word TF-IDF baseline | 50.00% | 49.15% |
| Multilingual embeddings | 41.67% | 40.01% |
| Character TF-IDF | 45.83% | 46.30% |
| Combined TF-IDF | 45.83% | 45.80% |

The original baseline achieved the highest validation
Macro F1 among the tested configurations.

Alternative models did not improve the validation score.

### Final Held-Out Test Results

- Accuracy: 50.00%
- Macro F1: 48.48%
- Correct predictions: 24 of 48

| Language | Accuracy | Macro F1 |
|---|---:|---:|
| Arabic | 37.50% | 35.87% |
| English | 62.50% | 60.89% |

The test sample is small, so these results should not
be generalized to larger real-world populations.

### Error Analysis

24 incorrect predictions were observed on the final
test set.

10 cases were manually reviewed and assigned
possible failure reasons, including:

- Dialectal Arabic
- Mixed sentiment
- Negation
- Label ambiguity
- Domain mismatch and non-review text
- Long or complex reviews

See results/error_analysis_detailed.csv.

### Clustering Insights

Four exploratory groups were created across the two
languages.

Cluster ar_1 had the highest observed negative
review rate (56.2%).

Cluster en_0 had the lowest (22.2%).

These rates come from original dataset labels,
not predictions on new customer feedback.

The clusters have not been validated as specific
customer issue categories.

## 5. How to Run

### Requirements

Python and the packages in requirements.txt.

A CUDA-compatible GPU is recommended for local
Qwen inference. CPU-only operation supports the
sentiment classifier and saved insights. The LLM
summary uses a deterministic fallback without a GPU.

### Setup

Clone this repository and enter the project folder:

```bash
git clone <YOUR_REPOSITORY_URL>
cd Smart-Customer-Feedback-Analyzer
pip install -r requirements.txt
python app.py
```

Replace YOUR_REPOSITORY_URL with the actual repository
URL after uploading the project.

Open the local Gradio URL shown in the terminal.

The first LLM request may take longer because its
pretrained model weights must be downloaded.

### Google Colab

The project notebook documents data preparation,
training, model comparison, clustering, and evaluation.

## 6. Sample Inputs and Outputs

Example 1:

Input: The customer service was terrible.

Observed output: Negative.

Example 2:

Input: الخدمة ممتازة والتعامل رائع

Observed output: Positive.

These are illustrative observed predictions.
The model can make errors, and probability scores
should not be interpreted as calibrated certainty.

The Customer Insights tab displays saved cluster
statistics and offers LLM-based summary generation.

## 7. Evaluation Files

The results directory includes:

- confusion_matrix.png
- language_evaluation.csv
- validation_by_language.csv
- model_comparison_by_language.csv
- test_predictions.csv
- error_analysis_detailed.csv
- customer_insights.csv
- insights_conclusion.txt
- llm_raw_summary.txt
- llm_verified_summary.txt

## 8. Limitations

- Small labeled training sample (150 records).
- Mixed source domains.
- Lower Arabic test performance.
- Weak exploratory cluster separation.
- Limited generalization to unseen customer feedback.
- Model probabilities are not calibrated.
- LLM output validation is rule-based and incomplete.
- GPU resources may be needed for live LLM inference.

## 9. Reproducibility

The repository provides source code, prepared artifacts,
model-selection results, and evaluation outputs.

Results were recorded from controlled train,
validation, and test splits.

Do not use the held-out test set to select new models.

## 10. Acknowledgments

This project was developed for the Applied NLP
Engineering Bootcamp.

SDAIA Academy:
https://github.com/SDAIAAcademy

Hugging Face:
https://huggingface.co/

The project is an individual educational prototype
and is not an official SDAIA Academy product.
"""

readme_path = PROJECT / "README.md"

readme_path.write_text(
    readme,
    encoding="utf-8"
)

print("README saved to:", readme_path)
print("Characters:", len(readme))

assert readme_path.exists()
assert readme_path.stat().st_size > 1000

print("\n✅ README.md created successfully!")


In [ ]:

# Step 34: Audit project files before GitHub upload

from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/Smart_Customer_Feedback_Analyzer"
)

required_files = [
    "app.py",
    "requirements.txt",
    "README.md",
    "models/sentiment_classifier.joblib",
    "results/customer_insights.csv",
    "results/confusion_matrix.png",
    "results/language_evaluation.csv",
    "results/model_comparison_by_language.csv",
    "results/error_analysis_detailed.csv",
    "results/insights_conclusion.txt",
]

print("===== GITHUB SUBMISSION FILE CHECK =====")

missing = []

for filename in required_files:
    path = PROJECT / filename
    if path.is_file():
        size = path.stat().st_size / 1024
        print(f"✅ {filename} ({size:.1f} KB)")
    else:
        print(f"❌ MISSING: {filename}")
        missing.append(filename)

print("\n===== NOTEBOOK CHECK =====")

notebooks = list(PROJECT.rglob("*.ipynb"))

if notebooks:
    for notebook in notebooks:
        print("✅", notebook.relative_to(PROJECT))
else:
    print("⚠️ Notebook has not been copied into project folder.")

print("\n===== README CHECK =====")

readme = (PROJECT / "README.md").read_text(
    encoding="utf-8"
)

print(
    "SDAIA Academy link:",
    "✅" if "github.com/SDAIAAcademy" in readme else "❌"
)

print(
    "Placeholder repository URL:",
    "⚠️ Needs updating"
    if "<YOUR_REPOSITORY_URL>" in readme
    else "✅ Not found"
)

print("\n===== SUMMARY =====")
print("Missing required files:", len(missing))
print("Notebook files:", len(notebooks))

print("\n✅ File audit completed!")
